## PostGIS Integration — Loading Wildfire Data & Building Spatial Queries

**Data:** CAL FIRE historical fire perimeters (`fire24_1.gdb`, 22,810 records) + NASA FIRMS active fire detections (VIIRS S-NPP, California, 2025)

**Database:** PostgreSQL + PostGIS 3.6 (`wildfire_db`, native Windows install)

**What this notebook does:**
- Connects to `wildfire_db` via SQLAlchemy engine and psycopg2 — the two Python interfaces to PostgreSQL
- Loads CAL FIRE fire perimeters from `fire24_1.gdb` into PostGIS table `fire_perimeters` using `gdf.to_postgis()` and creates a GiST spatial index for fast querying
- Downloads NASA FIRMS active fire detection CSV and loads it into PostGIS table `firms_fires`, creating point geometries from lat/lon columns using `ST_MakePoint` and a GiST spatial index
- Writes core spatial queries that will become FastAPI endpoints: `ST_Intersects` (FIRMS detections inside a fire perimeter), `ST_DWithin` (fire perimeters within N km of a coordinate), and top fires by acreage within a radius
- Builds a Python function `get_nearby_fires(lat, lon, radius_km)` that connects to PostGIS, runs a parameterized `ST_DWithin` query, and returns results as a GeoDataFrame via `gpd.read_postgis()`
- Visualizes query results with matplotlib to verify spatial correctness

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

password = os.getenv("DB_PASSWORD")
user = os.getenv("DB_USER")
host = os.getenv("DB_HOST")
dbname = os.getenv("DB_NAME")


In [2]:
import geopandas as gpd
import sqlalchemy as sa
from sqlalchemy import create_engine

In [3]:
# create geodataframe
gdf = gpd.read_file("../data/raw/fire24_1.gdb", layer="firep24_1")

f:\GeoPandas Projects\ca-wildfire-project\venv\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts. The processing may be really slow.  You can skip the processing by setting METHOD=SKIP, or only make it analyze counter-clock wise parts by setting METHOD=ONLY_CCW if you can assume that the outline of holes is counter-clock wise defined
  return ogr_read(


In [4]:
#Create SQLAlchemy engine
engine = create_engine(f"postgresql+psycopg2://{user}:{password}@{host}/{dbname}")


In [ ]:
# Rerun this cell only if it is first time running the code, or if you want to replace the existing table in the database.
# gdf.to_postgis("fire_perimeters", engine, if_exists="replace")

In [ ]:
#Verify data was written to PostGIS
import psycopg2
conn = psycopg2.connect(host=host, dbname=dbname, user=user, password=password)
cursor = conn.cursor()
cursor.execute("select * from fire_perimeters limit 5;")
results = cursor.fetchall()
print(results)

In [12]:
cursor.execute("CREATE INDEX idx_fire_perimeters_geom ON fire_perimeters USING GIST(geometry);")
conn.commit()


### Loading NASA FIRMS data
Example data downloaded was from NASA FIRMS for date of Malibu Fires (01/01/2025 - 03/31/2025)
Source: https://firms.modaps.eosdis.nasa.gov/

In [5]:
import pandas as pd

In [6]:
pd.read_csv("../data/raw/NASA_FIRMS/fire_archive_SV-C2_738489.csv")

,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_t31,frp,daynight,type
0,40.02568,-74.25027,297.86,0.48,0.40,2025-01-01,633,N,VIIRS,n,2,277.77,1.02,N,2
1,37.76586,-76.51810,295.04,0.38,0.43,2025-01-01,633,N,VIIRS,n,2,279.34,0.45,N,0
2,38.33913,-75.84859,301.90,0.54,0.42,2025-01-01,633,N,VIIRS,n,2,278.31,0.65,N,0
3,40.17477,-75.89853,298.00,0.57,0.43,2025-01-01,633,N,VIIRS,n,2,276.34,1.05,N,2
4,38.63375,-75.75109,305.35,0.54,0.42,2025-01-01,633,N,VIIRS,n,2,277.13,1.26,N,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
102843,19.40774,-155.27216,367.00,0.42,0.38,2025-03-31,2325,N,VIIRS,h,2,338.77,17.21,D,1
102844,19.40708,-155.27609,345.48,0.42,0.38,2025-03-31,2325,N,VIIRS,n,2,313.67,17.21,D,1
102845,19.40512,-155.28796,353.56,0.42,0.38,2025-03-31,2325,N,VIIRS,l,2,321.04,10.15,D,1
102846,19.40447,-155.29187,335.22,0.42,0.38,2025-03-31,2325,N,VIIRS,l,2,304.85,10.15,D,0
